In [0]:
spark.sql("USE fraud")

In [0]:
suspicious_df = spark.sql("""
SELECT *
FROM fraud.suspicious_transactions
""")

suspicious_df.display()

In [0]:
ai_analysis = spark.sql("""
SELECT *
FROM fraud.ai_fraud_analysis
""")

ai_analysis.display()

In [0]:
from pyspark.sql.functions import when

alerts_df = suspicious_df.withColumn(
    "alert_level",
    when(suspicious_df.risk_level == "HIGH","CRITICAL")
    .when(suspicious_df.risk_level == "MEDIUM","WARNING")
    .otherwise("LOW")
)

In [0]:
from pyspark.sql.functions import current_timestamp

alerts_df = alerts_df.withColumn("alert_time", current_timestamp())

In [0]:
alerts_df.write.format("delta").mode("overwrite").saveAsTable("fraud.fraud_alerts")

In [0]:
alerts_df = suspicious_df.join(ai_analysis, how="cross")

In [0]:
spark.sql("""
SELECT *
FROM fraud.fraud_alerts
LIMIT 20
""").display()